In [1]:
GRAY_TO_CLASS = {0: -1, 64: 0, 128: 1, 192: 2, 255: 3}  # -1 = background, ignorado
CLASS_TO_GRAY = {0: 64, 1: 128, 2: 192, 3: 255}
CLASS_NAMES = ["No_Proliferativo", "Proliferativo", "Esclerosado", "Excluido"]
NUM_CLASSES = 4

CONFIG = {
    # Paths — leer desde Entradas/ (tiles crudos + máscaras manuales)
    'images_dir': 'Entradas',              # <slide>/images/*.png — igual que U-Net
    'masks_dir':  'Entradas',              # <slide>/masks/*_mask.png — máscaras multiclase
    'output_dir': 'Salidas/Clasificador',
    'unet_checkpoint': 'Salidas/best_model.pth',  # para cargar channel_means/stds/reinhard

    # Split — MISMO que U-Net
    'train_size': 0.70,
    'val_size':   0.15,
    'seed':       42,

    # Crop/reconstruction
    'input_size':   224,
    'mask_size':    224,
    'margin_ratio': 0.25,
    'min_area_px':  1500,
    'min_distance': 15,

    # Model
    'backbone':    'efficientnet_b0',
    'input_mode':  'rgb_mask_manual',     # NEW: rgb_only | rgb_mask_manual | rgb_mask_unet | rgb_whitened
    'num_classes': 4,
    'pretrained':  True,

    # Training
    'max_epochs':      60,
    'batch_size':  32,
    'lr':          1e-3,
    'weight_decay': 1e-4,
    'warmup_epochs': 5,
    'use_amp':     True,
    'early_stopping_patience': 10,
    'early_stopping_min_delta': 1e-4,

    # Sampling & Loss strategy
    'sampling_strategy': 'sampler_balanced_ce_unweighted',  # NEW: sampler_balanced_ce_unweighted | sampler_normal_ce_weighted | focal_loss

    # Backbone ablation
    'backbone_ablation_list': [
        'efficientnet_b0',
        'resnet18',
        'resnet50',
        'googlenet',
        'mobilenetv3_large_100',
        'convnext_tiny',
    ],

    # Cross-validation
    'cv_mode': 'fixed',  # NEW: fixed | kfold | lobo
    'cv_k': 5,           # for kfold only

    # Augmentation adjustments for classification
    'elastic_p':     0.10,   # reducido vs. segmentación (0.25)
    'grid_dist_p':   0.10,   # reducido
    'mask_perturb_prob': 0.50,  # simula ruido de U-Net
}

In [2]:
# Standard library
import os
import json
import random
import warnings
from pathlib import Path
from typing import Tuple

# Scientific computing / ML
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import albumentations as A
import timm
import matplotlib.pyplot as plt

# Data loading and metrics
from PIL import Image, ImageFile
from scipy.ndimage import label as scipy_label, distance_transform_edt
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from skimage.feature import peak_local_max as _plm
from skimage.measure import regionprops
from skimage.segmentation import watershed as _watershed
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

# Allow PIL to load truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# --- CUDA performance tuning (T4 Tensor Cores) ---
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True          # auto-tune conv algorithms
    torch.backends.cuda.matmul.allow_tf32 = True   # TF32 for matmul (T4 supports it)
    torch.backends.cudnn.allow_tf32 = True         # TF32 for cudnn convolutions
    print(f"cudnn.benchmark: {torch.backends.cudnn.benchmark}")
    print(f"CUDA matmul TF32: {torch.backends.cuda.matmul.allow_tf32}")
    print(f"cudnn TF32: {torch.backends.cudnn.allow_tf32}")
    print(f"PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True")


# --- Preprocessing Transforms (Reinhard Normalization + Z-score) ---

class ReinhardNormalize:
    """Reinhard stain normalization in LAB color space for consistency across slides."""

    def __init__(self, target_stats: dict):
        self.target = target_stats

    @staticmethod
    def _get_tissue_mask(img_bgr: np.ndarray) -> np.ndarray:
        """Isolate tissue pixels from background and artifacts via luminance and saturation thresholds."""
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        mask = (lab[:, :, 0] < 230) & (hsv[:, :, 1] > 10)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
        mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        return mask.astype(bool)

    @staticmethod
    def compute_template_stats(image_paths: list, n_samples: int = 200) -> dict:
        """Compute median LAB statistics from tissue pixels to define normalization target."""
        paths_list = list(image_paths)[:n_samples]
        sample = random.sample(paths_list, min(n_samples, len(paths_list)))
        all_stats = []
        
        for p in sample:
            img = cv2.imread(str(p))
            if img is None:
                continue
            tissue = ReinhardNormalize._get_tissue_mask(img)
            if tissue.sum() < 100:
                continue
            lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
            stats = ([lab[..., c][tissue].mean() for c in range(3)] +
                     [lab[..., c][tissue].std()  for c in range(3)])
            all_stats.append(stats)
        
        if not all_stats:
            return {'mean_L': 50, 'mean_a': 128, 'mean_b': 128,
                    'std_L': 10, 'std_a': 10, 'std_b': 10}
        
        arr = np.array(all_stats)
        keys = ['mean_L', 'mean_a', 'mean_b', 'std_L', 'std_a', 'std_b']
        return {k: float(np.median(arr[:, i])) for i, k in enumerate(keys)}

    def __call__(self, img_bgr: np.ndarray) -> np.ndarray:
        """Normalize image to match template statistics, preserving background pixels."""
        tissue = self._get_tissue_mask(img_bgr)
        if tissue.sum() < 100:
            return img_bgr
        
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
        
        src = {c: (lab[..., i][tissue].mean(), lab[..., i][tissue].std())
               for i, c in enumerate(['L', 'a', 'b'])}
        
        result = lab.copy()
        for i, c in enumerate(['L', 'a', 'b']):
            m, s = src[c]
            result[..., i] = ((lab[..., i] - m) *
                              (self.target[f'std_{c}'] / (s + 1e-5)) +
                              self.target[f'mean_{c}'])
        
        result[~tissue] = lab[~tissue]
        result = np.clip(result, 0, 255).astype(np.uint8)
        return cv2.cvtColor(result, cv2.COLOR_LAB2BGR)


def compute_channel_stats(
    image_paths: list,
    n_samples: int = 200,
    reinhard_norm=None,
    std_floor: float = 0.03,
) -> Tuple[list, list]:
    """Compute weighted per-channel RGB stats from tissue pixels for Z-score normalization."""
    paths_list = list(image_paths)
    sample = random.sample(paths_list, min(n_samples, len(paths_list)))
    
    tile_means, tile_vars, tile_counts = [], [], []
    skipped_count = 0
    for p in sample:
        img_bgr = cv2.imread(str(p))
        if img_bgr is None:
            skipped_count += 1
            continue
        tissue = ReinhardNormalize._get_tissue_mask(img_bgr)
        if tissue.sum() < 100:
            skipped_count += 1
            continue

        if reinhard_norm is not None:
            img_bgr = reinhard_norm(img_bgr)

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        pixels = img_rgb[tissue]
        tile_means.append(pixels.mean(0))
        tile_vars.append(pixels.var(0))
        tile_counts.append(tissue.sum())
    
    if not tile_means:
        print(f"  ⚠️  WARNING: No valid tissue pixels found in {len(sample)} samples!")
        print(f"      {skipped_count} samples were skipped (tissue.sum() < 100 or read failed)")
        print(f"      Falling back to default values [0.5, 0.5, 0.5] and [0.2, 0.2, 0.2]")
        print(f"      This will make Z-score normalization INEFFECTIVE.")
        return [0.5, 0.5, 0.5], [0.2, 0.2, 0.2]
    
    print(f"  ✓ Computed stats from {len(tile_means)} valid samples (skipped {skipped_count})")
    
    means_arr = np.array(tile_means)
    vars_arr = np.array(tile_vars)
    counts_arr = np.array(tile_counts, dtype=np.float64)
    w = counts_arr / counts_arr.sum()
    
    mean = (means_arr * w[:, None]).sum(0)
    var = ((vars_arr + (means_arr - mean) ** 2) * w[:, None]).sum(0)
    std = np.sqrt(var)
    std = np.maximum(std, std_floor)
    return mean.tolist(), std.tolist()

def _collect_image_tiles(images_dir: str) -> list:
    """Collect input PNG tiles while excluding masks and generated mask files."""
    images_dir = Path(images_dir)
    image_paths = sorted(images_dir.glob('*/images/*.png'))

    # Fallback for flat/custom datasets: include PNGs except anything inside a masks folder
    # or files already named *_mask.png.
    if not image_paths:
        image_paths = sorted(
            p for p in images_dir.rglob('*.png')
            if 'masks' not in p.relative_to(images_dir).parts
            and not p.stem.endswith('_mask')
        )

    return image_paths


def _slide_name_from_image_path(image_path, images_dir) -> str:
    """Infer the biopsy/slide name from a tile path under <root>/<slide>/images/*.png."""
    image_path = Path(image_path)
    images_dir = Path(images_dir)
    try:
        rel = image_path.relative_to(images_dir)
        if len(rel.parts) >= 3 and rel.parts[1] == 'images':
            return rel.parts[0]
    except ValueError as e:
        warnings.warn(f'Path format issue for {image_path}: {e}')
        return None

    if image_path.parent.name == 'images' and image_path.parent.parent.name:
        return image_path.parent.parent.name
    return image_path.parent.name or image_path.stem


def _group_images_by_biopsy(image_paths: list, images_dir: str) -> dict:
    """Group image paths by biopsy/slide folder, supporting canonical and flat layouts."""
    images_dir = Path(images_dir)
    biopsias_dict = {}
    for img_path in image_paths:
        biopsia = _slide_name_from_image_path(img_path, images_dir)
        biopsias_dict.setdefault(biopsia, []).append(img_path)
    return biopsias_dict


def _safe_train_val_test_split(
    biopsias_list: list,
    train_size: float = 0.70,
    val_size: float = 0.15,
    seed: int = 42,
) -> Tuple[list, list, list]:
    """Split biopsy IDs without crashing on tiny datasets."""
    test_size = 1.0 - train_size - val_size
    assert test_size >= 0, "train_size + val_size must be <= 1.0"

    biopsias_list = list(biopsias_list)
    if not biopsias_list:
        return [], [], []

    # sklearn.train_test_split raises on very small lists. Keep deterministic, leak-free
    # biopsy-level splits and prefer having train data in quick/smoke-test datasets.
    if len(biopsias_list) == 1:
        return biopsias_list, [], []
    if len(biopsias_list) == 2:
        rng = random.Random(seed)
        shuffled = biopsias_list[:]
        rng.shuffle(shuffled)
        return [shuffled[0]], [shuffled[1]], []

    if test_size > 0:
        train_val_biopsias, test_biopsias = train_test_split(
            biopsias_list,
            test_size=test_size,
            random_state=seed,
        )
    else:
        train_val_biopsias = biopsias_list
        test_biopsias = []

    if val_size > 0 and len(train_val_biopsias) > 1:
        val_fraction = val_size / (train_size + val_size)
        train_biopsias, val_biopsias = train_test_split(
            train_val_biopsias,
            test_size=val_fraction,
            random_state=seed + 1,
        )
    else:
        train_biopsias = train_val_biopsias
        val_biopsias = []

    return train_biopsias, val_biopsias, test_biopsias


def split_biopsias(
    images_dir: str,
    train_size: float = 0.70,
    val_size: float = 0.15,
    seed: int = 42,
) -> Tuple[list, list, list, dict]:
    """Groups biopsias into train/val/test to prevent data leakage at slide level."""
    images_dir = Path(images_dir)
    all_images = _collect_image_tiles(images_dir)

    if not all_images:
        raise ValueError(f"No PNG image tiles found in {images_dir}. Expected files under */images/*.png")

    biopsias_dict = _group_images_by_biopsy(all_images, images_dir)
    train_biopsias, val_biopsias, test_biopsias = _safe_train_val_test_split(
        list(biopsias_dict.keys()),
        train_size=train_size,
        val_size=val_size,
        seed=seed,
    )

    return train_biopsias, val_biopsias, test_biopsias, biopsias_dict



def kfold_biopsy_split(biopsias: list, k: int = 5) -> list:
    """
    K-fold cross-validation split at biopsy level.
    Returns k folds, each with train/val/test biopsies.
    """
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    folds = []
    for train_idx, test_idx in kf.split(biopsias):
        train_b = [biopsias[i] for i in train_idx]
        val_b = train_b[:max(1, len(train_b)//5)]   # 20% of train for val
        train_b = train_b[len(val_b):]
        test_b = [biopsias[i] for i in test_idx]
        folds.append({'train': train_b, 'val': val_b, 'test': test_b})
    return folds


def leave_one_biopsy_out_split(biopsias: list) -> list:
    """
    Leave-one-biopsy-out cross-validation.
    Each biopsy is test once; rest divided 85/15 into train/val.
    """
    folds = []
    for i, test_b in enumerate(biopsias):
        remaining = [b for j, b in enumerate(biopsias) if j != i]
        val_b = remaining[:max(1, len(remaining)//6)]
        train_b = remaining[len(val_b):]
        folds.append({'train': train_b, 'val': val_b, 'test': [test_b]})
    return folds


def load_rgb_image(image_path: str) -> np.ndarray:
    """Load an RGB uint8 image with PIL tolerance for truncated files."""
    try:
        return np.array(Image.open(str(image_path)).convert('RGB'))
    except Exception as exc:
        raise RuntimeError(f"Failed to load image {image_path}: {exc}") from exc


def load_binary_mask(mask_path: str) -> np.ndarray:
    """Load a grayscale mask and binarize all non-zero classes as glomerulus."""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise RuntimeError(f"Failed to load mask: {mask_path}")
    return (mask > 0).astype(np.uint8)



def load_gray_mask(mask_path: str) -> np.ndarray:
    """Load a grayscale mask preserving multiclass values (0/64/128/192/255)."""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise RuntimeError(f"Failed to load mask: {mask_path}")
    return mask
def preprocess_rgb_image(
    img_rgb: np.ndarray,
    reinhard_norm=None,
    channel_means: list = None,
    channel_stds: list = None,
) -> np.ndarray:
    """Apply the same RGB -> Reinhard -> RGB/255 -> Z-score preprocessing used by the dataset."""
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    if reinhard_norm is not None:
        img_bgr = reinhard_norm(img_bgr)

    img_float = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    if channel_means is not None and channel_stds is not None:
        means = np.asarray(channel_means, dtype=np.float32)
        stds = np.asarray(channel_stds, dtype=np.float32)
        img_float = (img_float - means) / (stds + 1e-6)
    return img_float


def tensor_from_preprocessed_rgb(img_float: np.ndarray, device: torch.device = None) -> torch.Tensor:
    """Convert preprocessed HWC RGB float image to a BCHW float tensor."""
    tensor = torch.from_numpy(np.transpose(img_float, (2, 0, 1))).float().unsqueeze(0)
    return tensor.to(device) if device is not None else tensor


def make_mask_overlay(img_rgb: np.ndarray, mask_binary: np.ndarray, color=(255, 0, 0), alpha: float = 0.4) -> np.ndarray:
    """Blend a binary mask over an RGB image."""
    overlay = img_rgb.copy().astype(np.float32)
    overlay[mask_binary.astype(bool)] = color
    return cv2.addWeighted(img_rgb, 1 - alpha, overlay.astype(np.uint8), alpha, 0)


# ============================================================================
# Grayscale pixel values that correspond to glomerulus classes (any > 0 in practice)
# 64=No_Proliferativo, 128=Proliferativo, 192=Esclerosado, 255=Excluido/Excluyente
# Excluido (255) is INTENTIONALLY mapped to Glomerulus class 1 for binary segmentation
# ============================================================================
class GlomeruliDataset(Dataset):
    """Loads paired image-mask glomeruli tiles with online preprocessing and augmentation."""

    def __init__(
        self,
        images_dir: str,
        masks_dir: str = None,
        split: str = 'train',
        biopsias: list = None,
        biopsias_dict: dict = None,
        reinhard_norm=None,
        channel_means: list = None,
        channel_stds: list = None,
        train_size: float = 0.70,
        val_size: float = 0.15,
        seed: int = 42,
        transforms=None,
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir) if masks_dir is not None else Path(images_dir)
        self.split = split
        self.transforms = transforms
        self.reinhard_norm = reinhard_norm
        self.channel_means = channel_means if channel_means is not None else [0.5, 0.5, 0.5]
        self.channel_stds = channel_stds if channel_stds is not None else [0.2, 0.2, 0.2]

        assert split in {'train', 'val', 'test'}, f"Invalid split: {split}"
        assert self.images_dir.exists(), f"Images dir not found: {self.images_dir}"
        assert self.masks_dir.exists(), f"Masks dir not found: {self.masks_dir}"

        if biopsias is not None and biopsias_dict is not None:
            selected_biopsias = biopsias
            self.biopsias_dict = biopsias_dict
        else:
            train_biopsias, val_biopsias, test_biopsias, self.biopsias_dict = split_biopsias(
                self.images_dir,
                train_size=train_size,
                val_size=val_size,
                seed=seed,
            )
            selected_biopsias = {
                'train': train_biopsias,
                'val': val_biopsias,
                'test': test_biopsias,
            }[split]

        self.image_paths = []
        for biopsia in selected_biopsias:
            self.image_paths.extend(self.biopsias_dict.get(biopsia, []))

        self.image_paths = sorted(self.image_paths)

        paired = []
        missing = []
        for img_path in self.image_paths:
            mask_path = self._get_mask_path(img_path)
            if mask_path.exists():
                paired.append((img_path, mask_path))
            else:
                missing.append((img_path, mask_path))

        if missing:
            warnings.warn(
                f"Found {len(missing)} images without corresponding masks. "
                f"These will be skipped. First few: {missing[:3]}"
            )

        if not paired:
            raise ValueError("No valid image-mask pairs found after checking.")

        self.image_paths, self.mask_paths = zip(*paired)
        self.image_paths = list(self.image_paths)
        self.mask_paths = list(self.mask_paths)
        
        # Caches used by the train sampler/audits
        self._positive_flags = None
        self._tile_metadata = None
        self._annotation_tiles_by_key = None

    def _get_mask_path(self, image_path: Path) -> Path:
        """Convert image tile path to its mask path, supporting canonical and flat layouts."""
        rel = image_path.relative_to(self.images_dir)
        parts = list(rel.parts)
        stem = Path(parts[-1]).stem

        if len(parts) >= 3 and parts[1] == 'images':
            parts[1] = 'masks'
            parts[-1] = f"{stem}_mask.png"
            return self.masks_dir / Path(*parts)

        # Flat/custom fallback: first try <masks_dir>/<stem>_mask.png, then masks/<stem>_mask.png.
        flat_mask = self.masks_dir / f"{stem}_mask.png"
        if flat_mask.exists():
            return flat_mask
        return self.masks_dir / 'masks' / f"{stem}_mask.png"

    def get_positive_flags(self) -> list:
        """Return list of bools: True if tile mask contains at least one glomerulus pixel.
        
        Used by WeightedRandomSampler. Reads mask files once at dataset init time.
        Masks are small enough (1024x1024 uint8 = 1MB) that this is feasible.
        Caches result to avoid double scan.
        """
        if self._positive_flags is not None:
            return self._positive_flags
        
        flags = []
        for mask_path in self.mask_paths:
            try:
                flags.append(bool(np.any(load_binary_mask(mask_path))))
            except RuntimeError:
                flags.append(False)
        self._positive_flags = flags
        return flags

    def _load_annotation_tiles_by_key(self) -> dict:
        """Index per-slide annotations.json entries by (slide_folder, tile image path)."""
        if self._annotation_tiles_by_key is not None:
            return self._annotation_tiles_by_key

        tiles_by_key = {}
        for ann_path in sorted(self.images_dir.glob('*/annotations.json')):
            slide_folder = ann_path.parent.name
            try:
                with ann_path.open('r', encoding='utf-8') as f:
                    annotations = json.load(f)
            except Exception as exc:
                warnings.warn(f"Could not read annotations metadata from {ann_path}: {exc}")
                continue

            slide_name = annotations.get('slide') or slide_folder
            for tile in annotations.get('tiles', []):
                image_rel = tile.get('image')
                if not image_rel:
                    continue

                image_rel = str(Path(image_rel).as_posix())
                meta = dict(tile)
                meta['slide'] = slide_name
                meta['slide_folder'] = slide_folder
                meta['annotations_path'] = str(ann_path)

                # Support both the folder name and the slide name in case they differ.
                tiles_by_key[(slide_folder, image_rel)] = meta
                tiles_by_key[(slide_name, image_rel)] = meta

        self._annotation_tiles_by_key = tiles_by_key
        return tiles_by_key

    def get_tile_metadata(self, idx: int) -> dict:
        """Return annotations.json metadata for a dataset tile, or {} if unavailable."""
        if self._tile_metadata is None:
            tiles_by_key = self._load_annotation_tiles_by_key()
            metadata = []
            for image_path in self.image_paths:
                try:
                    rel = image_path.relative_to(self.images_dir)
                    slide_folder = rel.parts[0]
                    image_rel = Path(*rel.parts[1:]).as_posix()
                except Exception:
                    metadata.append({})
                    continue

                metadata.append(tiles_by_key.get((slide_folder, image_rel), {}))

            self._tile_metadata = metadata

        return self._tile_metadata[idx] if 0 <= idx < len(self._tile_metadata) else {}

    def get_sampling_weights(
        self,
        secondary_factor: float = 0.35,
        duplicate_aware: bool = True,
    ) -> Tuple[torch.Tensor, dict]:
        """Compute train-sampling weights with optional duplicate-aware glomerulus balancing."""
        positive_flags = self.get_positive_flags()
        n_positive = int(sum(positive_flags))
        n_negative = int(len(positive_flags) - n_positive)

        report = {
            'mode': 'simple',
            'n_positive': n_positive,
            'n_negative': n_negative,
            'num_unique_glomeruli': 0,
            'primary_links': 0,
            'secondary_links': 0,
            'unmatched_positive_tiles': 0,
        }

        if n_positive == 0 or n_negative == 0:
            return None, report

        def simple_weights(mode: str = 'simple'):
            report['mode'] = mode
            weight_pos = 1.0 / n_positive
            weight_neg = 1.0 / n_negative
            return torch.tensor(
                [weight_pos if flag else weight_neg for flag in positive_flags],
                dtype=torch.float32,
            ), report

        if not duplicate_aware:
            return simple_weights('simple')

        glomerulus_entries = {}
        for idx, is_positive in enumerate(positive_flags):
            if not is_positive:
                continue

            meta = self.get_tile_metadata(idx)
            glomeruli = meta.get('glomeruli') if meta else None
            if not glomeruli:
                continue

            slide = meta.get('slide') or meta.get('slide_folder') or _slide_name_from_image_path(
                self.image_paths[idx], self.images_dir
            )
            for glom in glomeruli:
                glom_id = glom.get('id')
                if glom_id is None:
                    continue

                role = str(glom.get('role', 'primary')).lower()
                try:
                    coverage = max(float(glom.get('coverage_pct', 0.0)), 0.0) / 100.0
                except (TypeError, ValueError):
                    coverage = 0.0

                role_factor = secondary_factor if role == 'secondary' else 1.0
                raw_weight = max(coverage, 1e-6) * role_factor
                key = (str(slide), str(glom_id))
                glomerulus_entries.setdefault(key, []).append((idx, raw_weight, role))

                if role == 'secondary':
                    report['secondary_links'] += 1
                else:
                    report['primary_links'] += 1

        if not glomerulus_entries:
            warnings.warn(
                "Duplicate-aware sampler requested, but no usable glomerulus metadata was found. "
                "Falling back to simple positive/negative balancing."
            )
            return simple_weights('fallback-simple-no-annotations')

        positive_mass = np.zeros(len(positive_flags), dtype=np.float64)
        for entries in glomerulus_entries.values():
            total = sum(raw for _, raw, _ in entries)
            if total <= 0:
                continue
            for idx, raw, _ in entries:
                positive_mass[idx] += raw / total

        unmatched_positive_tiles = 0
        for idx, is_positive in enumerate(positive_flags):
            if is_positive and positive_mass[idx] <= 0:
                # Keep mask-positive tiles with missing/partial metadata trainable.
                positive_mass[idx] = 1.0
                unmatched_positive_tiles += 1

        total_positive_mass = float(positive_mass.sum())
        if total_positive_mass <= 0:
            warnings.warn(
                "Duplicate-aware sampler produced zero positive mass. "
                "Falling back to simple positive/negative balancing."
            )
            return simple_weights('fallback-simple-zero-positive-mass')

        weights = np.zeros(len(positive_flags), dtype=np.float64)
        for idx, is_positive in enumerate(positive_flags):
            if is_positive:
                weights[idx] = positive_mass[idx] / total_positive_mass
            else:
                weights[idx] = 1.0 / n_negative

        report.update({
            'mode': 'duplicate-aware',
            'num_unique_glomeruli': len(glomerulus_entries),
            'unmatched_positive_tiles': unmatched_positive_tiles,
            'positive_weight_sum': float(weights[np.array(positive_flags, dtype=bool)].sum()),
            'negative_weight_sum': float(weights[~np.array(positive_flags, dtype=bool)].sum()),
            'secondary_factor': secondary_factor,
        })
        return torch.tensor(weights, dtype=torch.float32), report

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Load RGB tile -> Reinhard normalization -> augment -> Z-score -> tensors."""
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        img_rgb = load_rgb_image(img_path)
        mask_binary = load_binary_mask(mask_path)

        if self.reinhard_norm is not None:
            img_bgr = self.reinhard_norm(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        if self.transforms is not None:
            augmented = self.transforms(image=img_rgb, mask=mask_binary)
            img_rgb = augmented['image']
            mask_binary = augmented['mask']

        img_float = preprocess_rgb_image(
            img_rgb,
            reinhard_norm=None,  # already applied before augmentation when configured
            channel_means=self.channel_means,
            channel_stds=self.channel_stds,
        )
        img_tensor = torch.from_numpy(np.transpose(img_float, (2, 0, 1))).float()
        mask_tensor = torch.from_numpy(mask_binary.astype(np.int64)).long()

        return img_tensor, mask_tensor

# Augmentation hyperparameters (documented magic numbers)
AUGMENTATION_CONFIG = {
    'elastic': {'alpha': 120, 'sigma': 6.0},
    'he_stain': {
        'intensity_scale': (0.8, 1.2),
        'intensity_shift': (-0.1, 0.1),
    },
    'color_jitter': {
        'brightness': 0.2,
        'contrast': 0.2,
        'saturation': 0.2,
        'hue': 0.05,
    },
    'gaussian_noise': {'std_range': (0.01, 0.05)},
}

def get_transforms(size: int = 1024, config: dict | None = None):
    """
    Get training augmentations with tunable hyperparameters.
    
    Args:
        size: Image size
        config: Augmentation config dict. Defaults to AUGMENTATION_CONFIG
    
    Notes:
        Reinhard and Z-score normalization happen in GlomeruliDataset.__getitem__,
        not here. Train transforms run on RGB uint8 images before Z-score; validation
        uses an explicit no-op transform for symmetry.
    """
    if config is None:
        config = AUGMENTATION_CONFIG
    
    train_augment = A.Compose([
        # Geometric augmentations — applied to both image AND mask
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.75),
        A.Transpose(p=0.5),
        A.Rotate(limit=15, p=0.3),
        A.ShiftScaleRotate(scale_limit=0.15, rotate_limit=15, shift_limit=0.1, p=0.5),
        
        # Elastic deformations — simulate tissue preparation artifacts
        A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
        
        # H&E stain augmentation — simulates staining variability between biopsies
        # ImageOnlyTransform: applied to image only, mask is unchanged
        A.HEStain(
            method='random_preset',
            intensity_scale_range=(0.8, 1.2),
            intensity_shift_range=(-0.1, 0.1),
            augment_background=False,
            p=0.4,
        ),

        # Color augmentations — simulate stain variation between slides
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.3),
        
        # Noise — simulate scanner artifacts
        A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
        
        # Regularization via occlusion (use fill=128 to avoid NaN in BatchNorm)
        A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(32, 64),
            hole_width_range=(32, 64),
            fill=128,
            p=0.2,
        ),
    ], additional_targets={'mask': 'mask'})
    
    val_transform = A.Compose([])
    
    return train_augment, val_transform

# ===== CENTRALIZED POST-PROCESSING FUNCTION =====
# This function is used by threshold tuning, crop extraction, and reconstruction viz
# to ensure CONSISTENCY across all stages

def postprocess_prob_to_instances(
    prob_map,           # H×W float32 probability map (0-1)
    threshold=0.4,
    min_area_px=1500,
    min_distance=15,    # distance between watershed peaks
):
    """
    Consistent post-processing pipeline for all glomerulus detection tasks.
    
    Args:
        prob_map: H×W float32 array in [0, 1]
        threshold: decision threshold
        min_area_px: minimum instance area in pixels
        min_distance: minimum distance between watershed peaks
    
    Returns:
        list of skimage regionprops objects (instances)
    """
    # Step 1: Binary mask from probability
    binary_mask = (prob_map > threshold).astype(np.uint8)
    
    # Step 2: Morphological cleaning (close → open)
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel_close)
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel_open)
    
    # Step 3: Watershed-based instance separation
    if binary_mask.sum() < min_area_px:
        return []
    
    distance = distance_transform_edt(binary_mask)
    coords = _plm(distance, min_distance=min_distance, labels=binary_mask)
    
    if len(coords) == 0:
        # Single large region
        coords = np.array([[binary_mask.shape[0]//2, binary_mask.shape[1]//2]])
    
    marker_mask = np.zeros_like(binary_mask, dtype=bool)
    marker_mask[tuple(coords.T)] = True
    markers, _ = scipy_label(marker_mask)
    
    labels_map = _watershed(-distance, markers, mask=binary_mask)
    props = regionprops(labels_map)
    
    # Step 4: Filter by minimum area
    return [p for p in props if p.area >= min_area_px]


c:\Users\proyecto_final\Documents\Proyecto_Final_Glomerulos\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU: Tesla T4
VRAM: 17.00 GB
cudnn.benchmark: True
CUDA matmul TF32: True
cudnn TF32: True
PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True


In [3]:
def perturb_mask(binary_mask: np.ndarray) -> np.ndarray:
    """Perturba la máscara binaria para simular error de U-Net en entrenamiento."""
    ops = random.choices(['dilate', 'erode', 'close', 'open', 'shift', 'none'], k=1)[0]
    k = random.choice([3, 5])
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    if ops == 'dilate':
        return cv2.dilate(binary_mask.astype(np.uint8), kernel).astype(bool)
    elif ops == 'erode':
        return cv2.erode(binary_mask.astype(np.uint8), kernel).astype(bool)
    elif ops == 'close':
        return cv2.morphologyEx(binary_mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel).astype(bool)
    elif ops == 'open':
        return cv2.morphologyEx(binary_mask.astype(np.uint8), cv2.MORPH_OPEN, kernel).astype(bool)
    elif ops == 'shift':
        dx, dy = random.randint(-5, 5), random.randint(-5, 5)
        M = np.float32([[1, 0, dx], [0, 1, dy]])
        shifted = cv2.warpAffine(binary_mask.astype(np.uint8), M, binary_mask.shape[::-1])
        return shifted.astype(bool)
    return binary_mask

In [4]:
def build_glomerulus_manifest(
    images_dir: str,
    masks_dir: str,
    split_biopsias_result: dict,  # {'train': [...], 'val': [...], 'test': [...]}
    min_area_px: int = 1500,
    min_distance: int = 15,
    output_csv: str = None,
) -> pd.DataFrame:
    """
    Enumerate ALL glomerulus instances in the dataset (train+val+test).
    Each row is one glomerulus instance with its metadata, bbox, and class.
    
    Algorithm:
    1. For each split ('train', 'val', 'test'):
       - For each biopsia in split:
         - Enumerate tiles: {images_dir}/{biopsia}/images/*.png
         - For EACH tile:
           a. Load mask_gray = load_binary_mask(mask_path, binary=False)
              (PNG uint8 with values 0/64/128/192/255)
           b. Binarize: binary_mask = (mask_gray > 0).astype(float)
           c. Run instance detection: instances = postprocess_prob_to_instances(...)
           d. For EACH instance:
              - Extract bbox: instance.bbox → (r1, c1, r2, c2)
              - Extract region from mask_gray: mask_crop = mask_gray[r1:r2, c1:c2]
              - Find dominant (most frequent) value: dominant = bincount(mask_crop.flatten()).argmax()
              - Map to class: class_id = GRAY_TO_CLASS[dominant] (skip if -1/background)
              - Add row with: slide_id, tile_path, mask_path, instance_id, class_name, 
                             class_id, gray_value, bbox coords, split
    
    2. Return DataFrame with all rows
    3. If output_csv not None, save to CSV
    
    Args:
        images_dir: Root directory with {biopsia}/images/*.png structure
        masks_dir: Root directory with {biopsia}/masks/*_mask.png structure
        split_biopsias_result: dict with keys 'train', 'val', 'test' → list of biopsia names
        min_area_px: Minimum instance area in pixels (default 1500)
        min_distance: Minimum distance between watershed peaks (default 15)
        output_csv: If not None, save manifest to this CSV path
    
    Returns:
        pd.DataFrame with columns:
        - slide_id: biopsia name
        - tile_path: absolute path to tile PNG
        - mask_path: absolute path to mask PNG
        - instance_id: index within tile (0, 1, 2, ...)
        - class_name: CLASS_NAMES[class_id]
        - class_id: int 0-3
        - gray_value: dominant gray value (64, 128, 192, 255)
        - bbox_r1, bbox_c1, bbox_r2, bbox_c2: bounding box coordinates
        - split: 'train'/'val'/'test'
    
    Validation:
        - DataFrame has ~100s-1000s rows (1 per glomerulus, not per tile)
        - All columns present
        - No NaN in critical columns
        - Split distribution: ~70% train, ~15% val, ~15% test (approximate)
    """
    images_dir = Path(images_dir)
    masks_dir = Path(masks_dir)
    
    manifest_rows = []
    
    # Iterate through all splits
    for split_name in ['train', 'val', 'test']:
        biopsias = split_biopsias_result.get(split_name, [])
        
        for biopsia in biopsias:
            # Find all tiles in this biopsia: {images_dir}/{biopsia}/images/*.png
            tiles_glob = sorted((images_dir / biopsia / 'images').glob('*.png'))
            
            for tile_path in tiles_glob:
                # Construct corresponding mask path
                tile_stem = tile_path.stem
                mask_path = masks_dir / biopsia / 'masks' / f'{tile_stem}_mask.png'
                
                # Check if mask exists
                if not mask_path.exists():
                    warnings.warn(f"Mask not found for tile {tile_path}, skipping.")
                    continue
                
                # Load mask as grayscale (uint8, not binarized)
                try:
                    mask_gray = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                    if mask_gray is None:
                        warnings.warn(f"Failed to load mask: {mask_path}")
                        continue
                except Exception as e:
                    warnings.warn(f"Error loading mask {mask_path}: {e}")
                    continue
                
                # Binarize: any non-zero value is glomerulus
                binary_mask = (mask_gray > 0).astype(float)
                
                # Run instance detection
                instances = postprocess_prob_to_instances(
                    binary_mask,
                    threshold=0.5,
                    min_area_px=min_area_px,
                    min_distance=min_distance,
                )
                
                # For each detected instance, extract metadata
                for instance_id, instance in enumerate(instances):
                    # Extract bounding box
                    r1, c1, r2, c2 = instance.bbox
                    
                    # Get pixels for THIS instance only (using label_im directly)
                    # This ensures we use ONLY pixels from this instance, not bbox background
                    instance_pixels = mask_gray[label_im == lbl]
                    if len(instance_pixels) == 0:
                        continue
                    valid_pixels = instance_pixels[instance_pixels > 0]
                    if len(valid_pixels) == 0:
                        continue
                    dominant = int(np.bincount(valid_pixels).argmax())
                    
                    # Map to class
                    class_id = GRAY_TO_CLASS.get(dominant, -1)
                    
                    # Skip background/unknown classes
                    if class_id == -1:
                        continue
                    
                    class_name = CLASS_NAMES[class_id]
                    
                    # Add row to manifest
                    manifest_rows.append({
                        'slide_id': biopsia,
                        'tile_path': str(tile_path.absolute()),
                        'mask_path': str(mask_path.absolute()),
                        'instance_id': instance_id,
                        'class_name': class_name,
                        'class_id': class_id,
                        'gray_value': int(dominant),
                        'bbox_r1': int(r1),
                        'bbox_c1': int(c1),
                        'bbox_r2': int(r2),
                        'bbox_c2': int(c2),
                        'split': split_name,
                    })
    
    # Build DataFrame
    df = pd.DataFrame(manifest_rows)
    
    # Validation
    if df.empty:
        warnings.warn("No glomerulus instances found. Manifest is empty.")
        return df
    
    # Check for NaN in critical columns
    critical_cols = ['slide_id', 'tile_path', 'mask_path', 'instance_id', 
                     'class_id', 'gray_value', 'split']
    for col in critical_cols:
        if col not in df.columns:
            raise ValueError(f"Missing critical column: {col}")
        if df[col].isna().any():
            raise ValueError(f"Found NaN values in critical column: {col}")
    
    # Log statistics
    split_counts = df['split'].value_counts()
    print(f"\nGlomerulus Manifest Summary:")
    print(f"  Total instances: {len(df)}")
    print(f"  Split distribution:")
    for split in ['train', 'val', 'test']:
        count = split_counts.get(split, 0)
        pct = 100.0 * count / len(df) if len(df) > 0 else 0.0
        print(f"    {split}: {count} ({pct:.1f}%)")
    
    class_counts = df['class_id'].value_counts().sort_index()
    print(f"  Class distribution:")
    for class_id in sorted(class_counts.index):
        count = class_counts[class_id]
        pct = 100.0 * count / len(df)
        class_name = CLASS_NAMES[class_id]
        print(f"    {class_id} ({class_name}): {count} ({pct:.1f}%)")
    
    # Save to CSV if requested
    if output_csv is not None:
        output_path = Path(output_csv)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_path, index=False)
        print(f"\n  ✓ Manifest saved to {output_path}")
    
    return df


In [5]:
def validate_manifest(manifest_df: pd.DataFrame, output_dir: str = 'Salidas/Clasificador'):
    """
    Validate manifest: print distribution, warnings, and visual grid.
    """
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from matplotlib.gridspec import GridSpec
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # === 1. Distribution table ===
    print("\n=== MANIFEST DISTRIBUTION ===")
    print(f"Total instances: {len(manifest_df)}")
    
    # By class and split
    print("\nBy class and split:")
    dist_table = pd.crosstab(
        manifest_df['class_name'],
        manifest_df['split'],
        margins=True
    )
    print(dist_table)
    
    # By biopsy and class
    print("\nTop 10 biopsies by instance count:")
    biopsy_counts = manifest_df['slide_id'].value_counts().head(10)
    print(biopsy_counts)
    
    # === 2. Check for class imbalance warnings ===
    for split in ['train', 'val', 'test']:
        split_df = manifest_df[manifest_df['split'] == split]
        for cls in CLASS_NAMES:
            count = len(split_df[split_df['class_name'] == cls])
            if count == 0:
                print(f"⚠️  WARNING: {cls} has 0 instances in {split} split!")
            elif count < 5:
                print(f"⚠️  WARNING: {cls} has only {count} instances in {split} split")
    
    # === 3. Visual grid: N=8 crops per class ===
    fig = plt.figure(figsize=(20, 12))
    gs = GridSpec(len(CLASS_NAMES), 8, figure=fig, hspace=0.4, wspace=0.3)
    
    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_df = manifest_df[manifest_df['class_name'] == class_name].head(8)
        
        for crop_idx, (_, row) in enumerate(class_df.iterrows()):
            ax = fig.add_subplot(gs[class_idx, crop_idx])
            
            try:
                # Load RGB
                rgb = load_rgb_image(row['tile_path'])
                r1, c1, r2, c2 = int(row['bbox_r1']), int(row['bbox_c1']), int(row['bbox_r2']), int(row['bbox_c2'])
                crop_rgb = rgb[r1:r2, c1:c2]
                
                # Display
                ax.imshow(crop_rgb)
                ax.set_title(f"{class_name}\n{row['slide_id'][:15]}\narea={row.get('area_px', '?')}", fontsize=8)
                ax.axis('off')
            except Exception as e:
                ax.text(0.5, 0.5, f"Error: {str(e)[:20]}", ha='center', va='center', fontsize=8)
                ax.axis('off')
    
    # Save figure
    fig_path = output_dir / 'manifest_visual_grid.png'
    plt.savefig(fig_path, dpi=100, bbox_inches='tight')
    plt.close()
    print(f"\n✓ Saved visual grid to {fig_path}")
    
    # === 4. Compute area statistics ===
    if 'area_px' not in manifest_df.columns:
        # Compute area from bbox if not present
        manifest_df['area_px'] = (manifest_df['bbox_r2'] - manifest_df['bbox_r1']) * (manifest_df['bbox_c2'] - manifest_df['bbox_c1'])
    
    print("\nArea statistics (pixels) by class:")
    area_stats = manifest_df.groupby('class_name')['area_px'].agg(['min', 'mean', 'max', 'count'])
    print(area_stats)
    
    return manifest_df

In [6]:

def _crop_bounds_from_bbox(
    bbox: Tuple[int, int, int, int],
    image_shape: tuple,
    margin_ratio: float,
) -> Tuple[int, int, int, int]:
    """Return margin-expanded crop bounds clipped to an image/mask shape."""
    r1, c1, r2, c2 = map(int, bbox)
    height = max(1, r2 - r1)
    width = max(1, c2 - c1)
    margin = int(max(height, width) * margin_ratio)
    return (
        max(0, r1 - margin),
        max(0, c1 - margin),
        min(image_shape[0], r2 + margin),
        min(image_shape[1], c2 + margin),
    )


def _instance_mask_from_bbox(
    mask_gray: np.ndarray,
    bbox: Tuple[int, int, int, int],
    crop_bounds: Tuple[int, int, int, int],
) -> np.ndarray:
    """Build a binary mask for the manifest instance inside the margin crop.

    The manifest stores the regionprops bounding box, but not the original
    regionprops.image array. Reconstruct the instance support by taking the
    non-background pixels inside that bbox and placing them at the correct
    offset in the larger crop.
    """
    r1, c1, r2, c2 = map(int, bbox)
    r1m, c1m, r2m, c2m = map(int, crop_bounds)
    instance_mask = np.zeros((r2m - r1m, c2m - c1m), dtype=np.uint8)
    bbox_mask = (mask_gray[r1:r2, c1:c2] > 0).astype(np.uint8)
    rr1, cc1 = r1 - r1m, c1 - c1m
    instance_mask[rr1:rr1 + bbox_mask.shape[0], cc1:cc1 + bbox_mask.shape[1]] = bbox_mask
    return instance_mask


def _make_classifier_input(
    rgb: np.ndarray,
    binary_mask: np.ndarray,
    input_mode: str = 'rgb_mask_manual',
    unet_prob_map: np.ndarray = None,
) -> torch.Tensor:
    """
    Create classifier input tensor based on mode.
    
    Modes:
      - rgb_only: RGB only (3 channels)
      - rgb_whitened: RGB with background=255 (3 channels)
      - rgb_mask_manual: RGB + binary mask (4 channels)
      - rgb_mask_unet: RGB + U-Net soft probability (4 channels)
    """
    if input_mode == 'rgb_only':
        # RGB only, no masking
        tensor = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0
        return tensor
    
    elif input_mode == 'rgb_whitened':
        # RGB with background whitened (no mask channel)
        rgb_w = rgb.copy()
        rgb_w[binary_mask == 0] = 255
        tensor = torch.from_numpy(rgb_w).permute(2, 0, 1).float() / 255.0
        return tensor
    
    elif input_mode == 'rgb_mask_manual':
        # RGB + binary mask (current default)
        rgb_w = rgb.copy()
        rgb_w[binary_mask == 0] = 255
        tensor_rgb = torch.from_numpy(rgb_w).permute(2, 0, 1).float() / 255.0
        tensor_mask = torch.from_numpy(binary_mask[None, ...]).float()
        tensor = torch.cat([tensor_rgb, tensor_mask], dim=0)
        return tensor
    
    elif input_mode == 'rgb_mask_unet':
        # RGB + U-Net soft probability map (4 channels)
        assert unet_prob_map is not None, "unet_prob_map required for rgb_mask_unet mode"
        rgb_w = rgb.copy()
        rgb_w[binary_mask == 0] = 255
        tensor_rgb = torch.from_numpy(rgb_w).permute(2, 0, 1).float() / 255.0
        tensor_unet = torch.from_numpy(unet_prob_map[None, ...]).float()
        tensor = torch.cat([tensor_rgb, tensor_unet], dim=0)
        return tensor
    
    else:
        raise ValueError(f"Unknown input_mode: {input_mode}")

In [8]:

def get_classification_transforms(size: int = 224, config: dict = None):
    """Retorna (train_augment, val_transform) para clasificación."""
    config = config or {}
    elastic_p = config.get('elastic_p', 0.10)
    grid_dist_p = config.get('grid_dist_p', 0.10)

    train_aug = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Transpose(p=0.5),
        A.Rotate(limit=15, p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.3),
        A.ElasticTransform(alpha=60, sigma=3.0, p=elastic_p),
        A.GridDistortion(num_steps=5, distort_limit=0.15, p=grid_dist_p),
        A.HEStain(intensity_scale=(0.8, 1.2), intensity_shift=(-0.1, 0.1), p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.4),
        A.GaussNoise(std_range=(0.01, 0.05), p=0.3),
        A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.2),
    ], additional_targets={'mask': 'mask'})

    val_aug = A.Compose([])
    return train_aug, val_aug


def build_classifier(config: dict) -> nn.Module:
    """Build EfficientNet classifier with in_chans derived from input_mode."""
    input_mode = config.get('input_mode', 'rgb_mask_manual')
    in_chans = 3 if input_mode in ('rgb_only', 'rgb_whitened') else 4
    
    backbone = config.get('backbone', 'efficientnet_b0')
    
    model = timm.create_model(
        backbone,
        pretrained=config.get('pretrained', True),
        num_classes=config.get('num_classes', 4),
        in_chans=in_chans,
    )
    return model(backbone,
        pretrained=pretrained,
        num_classes=num_classes,
        in_chans=in_chans,
    )


In [9]:
# ============================================================================
# CELL 9: create_classification_dataloaders()
# ============================================================================

def create_classification_dataloaders(
    dataset_train: GlomeruliClassificationDataset,
    dataset_val: GlomeruliClassificationDataset,
    dataset_test: GlomeruliClassificationDataset,
    config: dict,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Create dataloaders with strategy-aware sampling."""
    strategy = config.get('sampling_strategy', 'sampler_balanced_ce_unweighted')
    batch_size = config.get('batch_size', 32)
    
    # Train dataloader with conditional sampler
    if strategy.startswith('sampler_balanced'):
        weights = dataset_train.get_sampling_weights()
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
        train_loader = DataLoader(
            dataset_train,
            sampler=sampler,
            batch_size=batch_size,
            pin_memory=True,
            num_workers=min(4, os.cpu_count()),
        )
    else:
        # No weighted sampler; use sequential
        train_loader = DataLoader(
            dataset_train,
            batch_size=batch_size,
            shuffle=True,
            pin_memory=True,
            num_workers=min(4, os.cpu_count()),
        )
    
    # Val and test dataloaders (no sampler)
    val_loader = DataLoader(
        dataset_val,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=True,
        num_workers=min(4, os.cpu_count()),
    )
    
    test_loader = DataLoader(
        dataset_test,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=True,
        num_workers=min(4, os.cpu_count()),
    )
    
    return train_loader, val_loader, test_loader

NameError: name 'GlomeruliClassificationDataset' is not defined

In [ ]:
# ============================================================================
# CELL 10: build_criterion()
# ============================================================================

class FocalLoss(nn.Module):
    """Focal Loss for addressing class imbalance."""
    def __init__(self, gamma: float = 2.0, weight: torch.Tensor = None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


def build_criterion(config: dict, train_dataset: GlomeruliClassificationDataset) -> nn.Module:
    """Build loss criterion based on sampling_strategy."""
    strategy = config.get('sampling_strategy', 'sampler_balanced_ce_unweighted')
    
    # Compute class weights
    weights = train_dataset.get_sampling_weights()
    # Convert sampling weights to loss weights: use inverse sqrt
    from collections import Counter
    counts = Counter(train_dataset.dataset['class_id'].values)
    class_counts = [counts.get(c, 1) for c in range(4)]
    loss_weights = torch.tensor([1.0 / (np.sqrt(c) + 1) for c in class_counts], dtype=torch.float32)
    loss_weights = loss_weights / loss_weights.sum() * len(class_counts)
    
    if strategy == 'sampler_balanced_ce_unweighted':
        # Balanced sampler + CE without class weights
        return nn.CrossEntropyLoss()
    
    elif strategy == 'sampler_normal_ce_weighted':
        # Normal sampler + CE with class weights
        return nn.CrossEntropyLoss(weight=loss_weights)
    
    elif strategy == 'focal_loss':
        # FocalLoss without class weights initially
        return FocalLoss(gamma=2.0, weight=None)
    
    else:
        raise ValueError(f"Unknown sampling_strategy: {strategy}")

In [ ]:

def train_epoch_clf(
    model,
    dataloader,
    criterion,
    optimizer,
    device,
    scaler=None,
    epoch=0,
    log_interval=20,
) -> float:
    """Entrena 1 epoch. Retorna loss promedio."""
    model.train()
    total_loss = 0.0
    num_batches = 0

    pbar = tqdm(dataloader, desc=f'Train Epoch {epoch}', leave=False)
    for batch_idx, (x, y) in enumerate(pbar):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with autocast():
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        total_loss += loss.item()
        num_batches += 1
        if batch_idx % log_interval == 0:
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    if num_batches == 0:
        raise ValueError("Cannot train on an empty dataloader.")
    return total_loss / num_batches


def eval_epoch_clf(
    model,
    dataloader,
    criterion,
    device,
    split_name='val',
) -> Tuple[float, dict]:
    """Evalúa 1 epoch. Retorna (loss_promedio, metrics_dict)."""
    from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix, precision_recall_fscore_support

    model.eval()
    total_loss = 0.0
    num_batches = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc=f'Eval {split_name}', leave=False)
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item()
            num_batches += 1
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(y.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    if num_batches == 0 or not all_labels:
        raise ValueError(f"Cannot evaluate '{split_name}' on an empty dataloader.")

    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)
    labels = list(range(NUM_CLASSES))

    balanced_acc = balanced_accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', labels=labels, zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average='weighted', labels=labels, zero_division=0)
    precision_per_class, recall_per_class, f1_per_class, support = precision_recall_fscore_support(
        all_labels, all_preds, labels=labels, average=None, zero_division=0
    )
    cm = confusion_matrix(all_labels, all_preds, labels=labels)

    return total_loss / num_batches, {
        'loss': total_loss / num_batches,
        'balanced_accuracy': balanced_acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'precision_per_class': precision_per_class,
        'recall_per_class': recall_per_class,
        'f1_per_class': f1_per_class,
        'support': support,
        'confusion_matrix': cm,
        'all_preds': all_preds,
        'all_labels': all_labels,
        'all_probs': all_probs,
    }


In [ ]:
def _train_single_fold(
    fold_name: str,
    fold_biopsias: dict,
    manifest_full: pd.DataFrame,
    config: dict,
    preprocessing_params: dict,
) -> dict:
    """
    Train classifier on a single fold.
    Returns dict with metrics.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    output_dir = Path(config['output_dir'])
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Filter manifest for this fold
    fold_mask = manifest_full['slide_id'].isin(fold_biopsias['train'] + fold_biopsias['val'])
    train_val_manifest = manifest_full[fold_mask].copy()
    
    # Further split into train/val within the fold
    train_mask = train_val_manifest['slide_id'].isin(fold_biopsias['train'])
    dataset_train = GlomeruliClassificationDataset(
        train_val_manifest[train_mask],
        config=config,
        split='train',
        **preprocessing_params,
    )
    dataset_val = GlomeruliClassificationDataset(
        train_val_manifest[~train_mask],
        config=config,
        split='val',
        **preprocessing_params,
    )
    
    # Test set (from fold_biopsias['test'])
    test_mask = manifest_full['slide_id'].isin(fold_biopsias['test'])
    dataset_test = GlomeruliClassificationDataset(
        manifest_full[test_mask],
        config=config,
        split='test',
        **preprocessing_params,
    )
    
    # Create dataloaders
    train_loader, val_loader, test_loader = create_classification_dataloaders(
        dataset_train, dataset_val, dataset_test, config
    )
    
    # Build model, optimizer, scheduler
    model = build_classifier(config).to(device)
    criterion = build_criterion(config, dataset_train)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.get('lr', 1e-3),
        weight_decay=config.get('weight_decay', 1e-4),
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=5,
        T_mult=2,
        eta_min=1e-6,
    )
    
    # Training loop
    max_epochs = config.get('max_epochs', 60)
    patience = config.get('early_stopping_patience', 10)
    best_balanced_acc = 0
    patience_counter = 0
    
    for epoch in range(max_epochs):
        train_loss = train_epoch_clf(model, train_loader, criterion, optimizer, device, config)
        val_loss, val_metrics = eval_epoch_clf(model, val_loader, criterion, device)
        val_balanced_acc = val_metrics['balanced_accuracy']
        
        if val_balanced_acc > best_balanced_acc:
            best_balanced_acc = val_balanced_acc
            patience_counter = 0
            # Save checkpoint
            ckpt_path = output_dir / f'best_model_{fold_name}.pth'
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break
        
        scheduler.step()
    
    # Final evaluation on test set
    model.load_state_dict(torch.load(output_dir / f'best_model_{fold_name}.pth'))
    test_loss, test_metrics = eval_epoch_clf(model, test_loader, criterion, device)
    
    return test_metrics

In [ ]:
def train_classifier(config: dict) -> Tuple[nn.Module, dict]:
    '''
    Función principal que orquesta entrenamiento completo del clasificador.
    
    1. Carga stats de U-Net (channel_means/stds/reinhard)
    2. Hace split de biopsias (igual que U-Net, seed=42)
    3. Construye manifest de instancias
    4. Crea dataloaders + modelo + optimizer + scheduler
    5. Entrena con early stopping sobre balanced_accuracy
    6. Guarda checkpoints
    7. Retorna modelo y reporte
    '''
    
    # Setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    output_dir = Path(config['output_dir'])
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Cargar stats de U-Net checkpoint
    print("[1/7] Loading U-Net checkpoint stats...")
    ckpt = torch.load(config['unet_checkpoint'], map_location='cpu')
    channel_means = ckpt.get('channel_means', [0.5, 0.5, 0.5])
    channel_stds = ckpt.get('channel_stds', [0.2, 0.2, 0.2])
    reinhard_stats = ckpt.get('reinhard_stats')
    print(f"  channel_means: {channel_means}, channel_stds: {channel_stds}")
    
    # 2. Split biopsias (MISMO que U-Net)
    print("[2/7] Splitting biopsias...")
    train_biopsias, val_biopsias, test_biopsias, _ = split_biopsias(
        config['images_dir'],
        train_size=config['train_size'],
        val_size=config['val_size'],
        seed=config['seed'],
    )
    biopsias_split = {
        'train': train_biopsias,
        'val': val_biopsias,
        'test': test_biopsias,
    }
    print(f"  Train: {len(biopsias_split['train'])} biopsias")
    print(f"  Val: {len(biopsias_split['val'])} biopsias")
    print(f"  Test: {len(biopsias_split['test'])} biopsias")
    
    # 3. Build manifest
    print("[3/7] Building glomerulus manifest...")
    manifest = build_glomerulus_manifest(
        images_dir=config['images_dir'],
        masks_dir=config['masks_dir'],
        split_biopsias_result=biopsias_split,
        min_area_px=config['min_area_px'],
        min_distance=config['min_distance'],
        output_csv=str(output_dir / 'manifest.csv'),
    )
    print(f"  Total instances: {len(manifest)}")
    if manifest.empty:
        raise ValueError("No glomerulus instances found; cannot train classifier.")
    print(f"  Class distribution:\n{manifest['class_id'].value_counts().sort_index()}")
    
    # 4. Reinhard normalization
    print("[4/7] Setting up preprocessing...")
    reinhard_norm = ReinhardNormalize(reinhard_stats) if reinhard_stats else None
    
    # 5. DataLoaders
    print("[5/7] Creating dataloaders...")
    train_loader, val_loader, test_loader = create_classification_dataloaders(
        manifest,
        config,
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
    )
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches: {len(val_loader)}")
    print(f"  Test batches: {len(test_loader)}")
    
    # 6. Model + Optimizer + Scheduler
    print("[6/7] Setting up model & optimizer...")
    model = build_classifier(
        backbone=config['backbone'],
        num_classes=config['num_classes'],
        in_chans=config['in_chans'],
        pretrained=config['pretrained'],
    ).to(device)
    
    criterion = build_criterion(manifest, device)
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay'],
    )
    
    # Scheduler: warmup lineal + cosine annealing
    warmup = LinearLR(
        optimizer,
        start_factor=0.1,
        end_factor=1.0,
        total_iters=config['warmup_epochs'],
    )
    cosine = CosineAnnealingLR(
        optimizer,
        T_max=max(1, config['epochs'] - config['warmup_epochs']),
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup, cosine],
        milestones=[config['warmup_epochs']],
    )
    
    scaler = GradScaler() if config['use_amp'] else None
    
    # 7. Training loop
    print("[7/7] Training...")
    best_val_balanced_acc = -1
    patience_counter = 0
    history = {
        'train_loss': [], 'val_loss': [],
        'val_balanced_acc': [], 'val_macro_f1': [],
    }
    
    for epoch in range(config['epochs']):
        # Train
        train_loss = train_epoch_clf(
            model, train_loader, criterion, optimizer, device,
            scaler=scaler, epoch=epoch,
        )
        
        # Validate
        val_loss, val_metrics = eval_epoch_clf(
            model, val_loader, criterion, device, split_name='val'
        )
        
        val_balanced_acc = val_metrics['balanced_accuracy']
        val_macro_f1 = val_metrics['macro_f1']
        
        # Logging
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_balanced_acc'].append(val_balanced_acc)
        history['val_macro_f1'].append(val_macro_f1)
        
        print(f"Epoch {epoch+1:3d}/{config['epochs']} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Bal.Acc: {val_balanced_acc:.4f} | "
              f"Val Macro F1: {val_macro_f1:.4f}")
        
        # Early stopping + checkpoint
        if val_balanced_acc > best_val_balanced_acc + config['early_stopping_min_delta']:
            best_val_balanced_acc = val_balanced_acc
            patience_counter = 0
            
            # Guardar best checkpoint
            ckpt_path = output_dir / 'best_classifier.pth'
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'scaler_state_dict': scaler.state_dict() if scaler else None,
                'best_val_balanced_acc': best_val_balanced_acc,
                'class_names': CLASS_NAMES,
                'channel_means': channel_means,
                'channel_stds': channel_stds,
                'reinhard_stats': reinhard_stats,
                'config': config,
            }, ckpt_path)
            print(f"  → Checkpoint saved to {ckpt_path}")
        else:
            patience_counter += 1
            if patience_counter >= config['early_stopping_patience']:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break
        
        scheduler.step()
    
    # Cargar el mejor checkpoint antes de evaluar test y retornar el modelo
    best_ckpt_path = output_dir / 'best_classifier.pth'
    if best_ckpt_path.exists():
        best_ckpt = torch.load(best_ckpt_path, map_location=device)
        model.load_state_dict(best_ckpt['model_state_dict'])

    # Evaluate on test set with the best validation model
    print("\nEvaluating on test set...")
    test_loss, test_metrics = eval_epoch_clf(
        model, test_loader, criterion, device, split_name='test'
    )
    
    # Reporte final
    report = {
        'history': history,
        'test_metrics': test_metrics,
        'test_loss': test_loss,
        'best_val_balanced_acc': best_val_balanced_acc,
    }
    
    return model, report


NameError: name 'Tuple' is not defined

In [ ]:

def evaluate_classifier(
    model,
    test_loader,
    device,
    class_names: list = None,
) -> dict:
    """Evalúa el clasificador en test set con reportes detallados."""
    from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix
    from sklearn.metrics import precision_recall_fscore_support, classification_report
    import seaborn as sns

    if class_names is None:
        class_names = CLASS_NAMES

    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for x, y in tqdm(test_loader, desc='Evaluating', leave=False):
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(y.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    if not all_labels:
        raise ValueError("Cannot evaluate classifier on an empty test_loader.")

    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)
    labels = list(range(NUM_CLASSES))

    balanced_acc = balanced_accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', labels=labels, zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average='weighted', labels=labels, zero_division=0)
    precision_per_class, recall_per_class, f1_per_class, support = precision_recall_fscore_support(
        all_labels, all_preds, labels=labels, average=None, zero_division=0
    )
    cm = confusion_matrix(all_labels, all_preds, labels=labels)

    try:
        auc_ovr = roc_auc_score(all_labels, all_probs, multi_class='ovr', labels=labels)
    except ValueError:
        auc_ovr = np.nan

    class_report = classification_report(
        all_labels, all_preds, labels=labels, target_names=class_names, zero_division=0
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title('Confusion Matrix')
    axes[0].set_ylabel('True')
    axes[0].set_xlabel('Predicted')

    x = np.arange(len(class_names))
    width = 0.25
    axes[1].bar(x - width, precision_per_class, width, label='Precision')
    axes[1].bar(x, recall_per_class, width, label='Recall')
    axes[1].bar(x + width, f1_per_class, width, label='F1')
    axes[1].set_ylabel('Score')
    axes[1].set_title('Per-Class Metrics')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(class_names, rotation=45, ha='right')
    axes[1].legend()
    axes[1].set_ylim([0, 1])
    plt.tight_layout()
    plt.show()

    print("\n" + "="*60)
    print("EVALUATION REPORT")
    print("="*60)
    print(f"\nGlobal Metrics:")
    print(f"  Balanced Accuracy: {balanced_acc:.4f}")
    print(f"  Macro F1:          {macro_f1:.4f}")
    print(f"  Weighted F1:       {weighted_f1:.4f}")
    print(f"  AUC OvR:           {auc_ovr:.4f}")
    print(f"\nPer-Class Metrics:")
    for name, prec, rec, f1, sup in zip(class_names, precision_per_class, recall_per_class, f1_per_class, support):
        print(f"  {name:20s}: P={prec:.3f} R={rec:.3f} F1={f1:.3f} (n={int(sup)})")
    print(f"\n{class_report}")
    print("="*60)

    return {
        'balanced_accuracy': balanced_acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'auc_ovr': auc_ovr,
        'precision_per_class': precision_per_class,
        'recall_per_class': recall_per_class,
        'f1_per_class': f1_per_class,
        'support': support,
        'confusion_matrix': cm,
        'all_preds': all_preds,
        'all_labels': all_labels,
        'all_probs': all_probs,
        'classification_report': class_report,
    }


In [ ]:

def export_multiclass_mask(
    tile_rgb_path: str,
    classifier_model,
    unet_prob_map: np.ndarray,
    unet_threshold: float,
    device,
    config: dict,
    reinhard_norm,
    channel_means: list,
    channel_stds: list,
    output_dir: str = None,
) -> Tuple[np.ndarray, list]:
    """Clasifica instancias en un tile y exporta una máscara PNG multiclase."""
    tile_rgb = load_rgb_image(tile_rgb_path)
    instances = postprocess_prob_to_instances(
        unet_prob_map,
        threshold=unet_threshold,
        min_area_px=config['min_area_px'],
        min_distance=config['min_distance'],
    )

    pred_mask_gray = np.zeros(tile_rgb.shape[:2], dtype=np.uint8)
    instances_report = []
    classifier_model.eval()

    with torch.no_grad():
        for inst_id, instance in enumerate(instances):
            r1, c1, r2, c2 = map(int, instance.bbox)
            crop_bounds = _crop_bounds_from_bbox((r1, c1, r2, c2), tile_rgb.shape[:2], config['margin_ratio'])
            r1m, c1m, r2m, c2m = crop_bounds

            img_crop = tile_rgb[r1m:r2m, c1m:c2m].copy()
            mask_crop = np.zeros((r2m - r1m, c2m - c1m), dtype=np.uint8)
            rr1, cc1 = r1 - r1m, c1 - c1m
            instance_mask = instance.image.astype(np.uint8)
            mask_crop[rr1:rr1 + instance_mask.shape[0], cc1:cc1 + instance_mask.shape[1]] = instance_mask

            x = _make_classifier_input(
                img_crop,
                mask_crop,
                config['input_size'],
                reinhard_norm=reinhard_norm,
                channel_means=channel_means,
                channel_stds=channel_stds,
                transforms=None,
                background_value=255,
            ).to(device)[None]

            logits = classifier_model(x)
            probs = torch.softmax(logits, dim=1)
            pred_class_id = torch.argmax(logits, dim=1).item()
            pred_prob = probs[0, pred_class_id].item()
            gray_value = CLASS_TO_GRAY[pred_class_id]

            pred_region = pred_mask_gray[r1:r2, c1:c2]
            pred_region[instance_mask > 0] = gray_value

            instances_report.append({
                'instance_id': inst_id,
                'class_id': pred_class_id,
                'class_name': CLASS_NAMES[pred_class_id],
                'gray_value': gray_value,
                'probability': pred_prob,
                'bbox': (r1, c1, r2, c2),
                'area_px': int(np.sum(instance_mask > 0)),
            })

    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        png_path = output_dir / f'{Path(tile_rgb_path).stem}_classified.png'
        cv2.imwrite(str(png_path), pred_mask_gray)
        csv_path = output_dir / f'{Path(tile_rgb_path).stem}_instances.csv'
        pd.DataFrame(instances_report).to_csv(csv_path, index=False)

    return pred_mask_gray, instances_report


In [ ]:
def compare_backbones(
    manifest_df: pd.DataFrame,
    config: dict,
    preprocessing_params: dict,
) -> dict:
    """
    Compare multiple backbones from backbone_ablation_list.
    Returns dict with results for each backbone.
    """
    backbones = config.get('backbone_ablation_list', ['efficientnet_b0'])
    results = {}
    
    fold_biopsies = {
        'train': config.get('train_biopsies', []),
        'val': config.get('val_biopsies', []),
        'test': config.get('test_biopsies', []),
    }
    
    for backbone in backbones:
        print(f"\n{'='*60}")
        print(f"Testing backbone: {backbone}")
        print(f"{'='*60}")
        
        cfg = {**config, 'backbone': backbone, 'max_epochs': 30}
        metrics = _train_single_fold('backbone_test', fold_biopsies, manifest_df, cfg, preprocessing_params)
        results[backbone] = metrics
        
        print(f"  macro_F1: {metrics.get('macro_f1', 'N/A'):.4f}")
        print(f"  balanced_acc: {metrics.get('balanced_accuracy', 'N/A'):.4f}")
    
    # Print summary
    print(f"\n{'='*60}")
    print("SUMMARY")
    print(f"{'='*60}")
    print(f"{'Backbone':<30} {'macro_F1':<12} {'balanced_acc':<12}")
    for backbone, metrics in results.items():
        print(f"{backbone:<30} {metrics.get('macro_f1', 0):<12.4f} {metrics.get('balanced_accuracy', 0):<12.4f}")
    
    return results

In [ ]:
def compare_input_modes(
    manifest_df: pd.DataFrame,
    config: dict,
    preprocessing_params: dict,
) -> dict:
    """
    Compare 3 input modes: rgb_only, rgb_whitened, rgb_mask_manual.
    Returns dict with results for each mode.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    modes = ['rgb_only', 'rgb_whitened', 'rgb_mask_manual']
    results = {}
    
    fold_biopsies = {
        'train': config.get('train_biopsies', []),
        'val': config.get('val_biopsies', []),
        'test': config.get('test_biopsies', []),
    }
    
    for mode in modes:
        print(f"\n{'='*60}")
        print(f"Testing input_mode: {mode}")
        print(f"{'='*60}")
        
        cfg = {**config, 'input_mode': mode, 'max_epochs': 30}
        metrics = _train_single_fold('mode_test', fold_biopsies, manifest_df, cfg, preprocessing_params)
        results[mode] = metrics
        
        print(f"  macro_F1: {metrics.get('macro_f1', 'N/A'):.4f}")
        print(f"  balanced_acc: {metrics.get('balanced_accuracy', 'N/A'):.4f}")
    
    # Print summary
    print(f"\n{'='*60}")
    print("SUMMARY")
    print(f"{'='*60}")
    print(f"{'Mode':<25} {'macro_F1':<12} {'balanced_acc':<12}")
    for mode, metrics in results.items():
        print(f"{mode:<25} {metrics.get('macro_f1', 0):<12.4f} {metrics.get('balanced_accuracy', 0):<12.4f}")
    
    return results

In [ ]:
def compare_sampling_strategies(
    manifest_df: pd.DataFrame,
    config: dict,
    preprocessing_params: dict,
) -> dict:
    """
    Compare 3 sampling strategies: sampler_balanced_ce_unweighted, sampler_normal_ce_weighted, focal_loss.
    Returns dict with results for each strategy.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    strategies = [
        'sampler_balanced_ce_unweighted',
        'sampler_normal_ce_weighted',
        'focal_loss',
    ]
    results = {}
    
    for strategy in strategies:
        print(f"\n{'='*60}")
        print(f"Testing strategy: {strategy}")
        print(f"{'='*60}")
        
        cfg = {**config, 'sampling_strategy': strategy, 'max_epochs': 30}
        
        # Use a simple fixed split
        fold_biopsies = {
            'train': config.get('train_biopsies', []),
            'val': config.get('val_biopsies', []),
            'test': config.get('test_biopsies', []),
        }
        
        metrics = _train_single_fold('strategy_test', fold_biopsies, manifest_df, cfg, preprocessing_params)
        results[strategy] = metrics
        
        print(f"  macro_F1: {metrics.get('macro_f1', 'N/A'):.4f}")
        print(f"  balanced_acc: {metrics.get('balanced_accuracy', 'N/A'):.4f}")
        print(f"  recall_Esclerosado: {metrics.get('recall_Esclerosado', 'N/A'):.4f}")
        print(f"  recall_Excluido: {metrics.get('recall_Excluido', 'N/A'):.4f}")
    
    # Print summary table
    print(f"\n{'='*60}")
    print("SUMMARY")
    print(f"{'='*60}")
    print(f"{'Strategy':<35} {'macro_F1':<12} {'balanced_acc':<12} {'recall_Escl':<12} {'recall_Excl':<12}")
    for strategy, metrics in results.items():
        print(f"{strategy:<35} {metrics.get('macro_f1', 0):<12.4f} "
              f"{metrics.get('balanced_accuracy', 0):<12.4f} "
              f"{metrics.get('recall_Esclerosado', 0):<12.4f} "
              f"{metrics.get('recall_Excluido', 0):<12.4f}")
    
    return results

In [ ]:

def evaluate_pipeline(
    test_manifest: pd.DataFrame,
    classifier_model,
    device,
    config: dict,
    reinhard_norm,
    channel_means: list,
    channel_stds: list,
) -> dict:
    """Evalúa clasificación de instancias GT y reporta IoU/Dice por clase en crops."""
    classifier_model.eval()

    def compute_iou_per_class(pred, gt, num_classes):
        iou_per_class = []
        for c in range(num_classes):
            pred_c = pred == c
            gt_c = gt == c
            intersection = np.sum(pred_c & gt_c)
            union = np.sum(pred_c | gt_c)
            iou_per_class.append(intersection / union if union > 0 else np.nan)
        return iou_per_class

    def compute_dice_per_class(pred, gt, num_classes):
        dice_per_class = []
        for c in range(num_classes):
            pred_c = pred == c
            gt_c = gt == c
            intersection = np.sum(pred_c & gt_c)
            denom = np.sum(pred_c) + np.sum(gt_c)
            dice_per_class.append(2 * intersection / denom if denom > 0 else np.nan)
        return dice_per_class

    test_df = test_manifest[test_manifest['split'] == 'test'].reset_index(drop=True)
    if test_df.empty:
        raise ValueError("No test instances found in manifest; cannot evaluate pipeline.")

    all_ious = {c: [] for c in range(NUM_CLASSES)}
    all_dices = {c: [] for c in range(NUM_CLASSES)}

    with torch.no_grad():
        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Evaluating pipeline'):
            tile_rgb = load_rgb_image(row['tile_path'])
            mask_gt = load_gray_mask(row['mask_path'])
            bbox = (row['bbox_r1'], row['bbox_c1'], row['bbox_r2'], row['bbox_c2'])
            r1m, c1m, r2m, c2m = _crop_bounds_from_bbox(bbox, tile_rgb.shape[:2], config['margin_ratio'])

            img_crop = tile_rgb[r1m:r2m, c1m:c2m].copy()
            mask_binary = _instance_mask_from_bbox(mask_gt, bbox, (r1m, c1m, r2m, c2m))

            x = _make_classifier_input(
                img_crop,
                mask_binary,
                config['input_size'],
                reinhard_norm=reinhard_norm,
                channel_means=channel_means,
                channel_stds=channel_stds,
                transforms=None,
                background_value=255,
            ).to(device)[None]

            logits = classifier_model(x)
            pred_class_id = torch.argmax(logits, dim=1).item()

            mask_resized = cv2.resize(mask_binary, (config['input_size'], config['input_size']), interpolation=cv2.INTER_NEAREST)
            pred_mask_class = np.zeros_like(mask_resized, dtype=np.uint8)
            pred_mask_class[mask_resized > 0] = pred_class_id

            if 'class_id' in row:
                gt_class_id = int(row['class_id'])
            else:
                valid = mask_gt[mask_gt > 0]
                gt_class_id = GRAY_TO_CLASS.get(int(np.bincount(valid).argmax()), 0) if len(valid) else 0

            gt_mask_class = np.zeros_like(mask_resized, dtype=np.uint8)
            gt_mask_class[mask_resized > 0] = max(0, gt_class_id)

            iou_per_class = compute_iou_per_class(pred_mask_class, gt_mask_class, NUM_CLASSES)
            dice_per_class = compute_dice_per_class(pred_mask_class, gt_mask_class, NUM_CLASSES)
            for c in range(NUM_CLASSES):
                if not np.isnan(iou_per_class[c]):
                    all_ious[c].append(iou_per_class[c])
                if not np.isnan(dice_per_class[c]):
                    all_dices[c].append(dice_per_class[c])

    miou_per_class = {c: float(np.mean(all_ious[c])) if all_ious[c] else np.nan for c in range(NUM_CLASSES)}
    dice_per_class = {c: float(np.mean(all_dices[c])) if all_dices[c] else np.nan for c in range(NUM_CLASSES)}
    miou_overall = float(np.nanmean(list(miou_per_class.values())))
    dice_overall = float(np.nanmean(list(dice_per_class.values())))

    print("\n" + "="*60)
    print("PIPELINE EVALUATION (Predictions vs Ground Truth)")
    print("="*60)
    print(f"\nOverall mIoU: {miou_overall:.4f}")
    print(f"Overall Dice: {dice_overall:.4f}")
    print(f"\nPer-Class Metrics:")
    for c in range(NUM_CLASSES):
        print(f"  {CLASS_NAMES[c]:20s}: mIoU={miou_per_class[c]:.4f} Dice={dice_per_class[c]:.4f}")
    print("="*60)

    return {
        'miou_overall': miou_overall,
        'dice_overall': dice_overall,
        'miou_per_class': miou_per_class,
        'dice_per_class': dice_per_class,
    }
